In [15]:
SAMPLE_HZ = 200
WINDOW_SAMPLES = 200
ACCELERATION_CHANNELS = ["ax", "ay", "az"]
GYROSCOPE_CHANNELS = ["gx", "gy", "gz"]
IMU_CHANNELS = ACCELERATION_CHANNELS + GYROSCOPE_CHANNELS
MODEL_INPUT_SIZE = WINDOW_SAMPLES * len(IMU_CHANNELS)

# Load the long-form data and keep fall activities for fall-type classification.
data = load_sisfall(PARQUET_DIR)
fall_data = data[data["activity"].str.startswith("F")].copy()

# Create the split at trial level so samples from one trial do not leak across sets.
trial_index = (
    fall_data[["trial_id", "activity", "subject"]]
    .drop_duplicates()
    .sort_values("trial_id")
    .reset_index(drop=True)
)

train_trials, test_trials = train_test_split(
    trial_index,
    test_size=0.20,
    random_state=42,
    stratify=trial_index["activity"],
)

train_ids = set(train_trials["trial_id"])
test_ids = set(test_trials["trial_id"])
train_data = fall_data[fall_data["trial_id"].isin(train_ids)].copy()
test_data = fall_data[fall_data["trial_id"].isin(test_ids)].copy()

# Map the 15 fall activity codes to supervised integer class labels.
fall_classes = sorted(fall_data["activity"].unique().tolist())
activity_to_label = {activity: label for label, activity in enumerate(fall_classes)}
label_to_activity = {label: activity for activity, label in activity_to_label.items()}
train_data["label"] = train_data["activity"].map(activity_to_label)
test_data["label"] = test_data["activity"].map(activity_to_label)

print("Sampling rate:", SAMPLE_HZ, "Hz")
print("Window length:", WINDOW_SAMPLES, "samples (1 second)")
print("IMU channels:", IMU_CHANNELS)
print("PyTorch input width:", MODEL_INPUT_SIZE)
print("Fall trials:", len(trial_index))
print("Train trials:", len(train_trials), "| test trials:", len(test_trials))
print("Train/test trial overlap:", len(train_ids & test_ids))
print("Classes:", len(fall_classes), fall_classes)
print("Train rows:", len(train_data), "| test rows:", len(test_data))

Sampling rate: 200 Hz
Window length: 200 samples (1 second)
IMU channels: ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
PyTorch input width: 1200
Fall trials: 1798
Train trials: 1438 | test trials: 360
Train/test trial overlap: 0
Classes: 15 ['F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09', 'F10', 'F11', 'F12', 'F13', 'F14', 'F15']
Train rows: 4313741 | test rows: 1079973


## Load and create the split

In [16]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from ysafetyml.sisfall import PROJECT_ROOT, load_sisfall

PARQUET_DIR = PROJECT_ROOT / "data" / "sisfall_parquet"

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.15.0+cu132
CUDA available: True


# SisFall Supervised Learning Setup

Load the SisFall Parquet data and create a trial-level train/test split. The planned PyTorch input is **600 values per window**: 200 accelerometer-X samples, 200 Y samples, and 200 Z samples.

In [17]:
class Conv1DBlock(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 7, stride: int = 2):
        super().__init__()
        padding = kernel_size // 2
        self.block = torch.nn.Sequential(
            torch.nn.Conv1d(in_channels, out_channels, kernel_size, stride=stride, padding=padding),
            torch.nn.BatchNorm1d(out_channels),
            torch.nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class FallConvNet(torch.nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = torch.nn.Sequential(
            Conv1DBlock(len(IMU_CHANNELS), 32, kernel_size=7, stride=2), # 200 -> 100 samples
            Conv1DBlock(32, 64, kernel_size=5, stride=2),                 # 100 -> 50 samples
            Conv1DBlock(64, 128, kernel_size=3, stride=2),                # 50 -> 25 samples
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.AdaptiveAvgPool1d(1),
            torch.nn.Flatten(),
            torch.nn.Linear(128, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.25),
            torch.nn.Linear(64, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return self.classifier(x)

In [18]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 500
LEARNING_RATE = 1e-3
BATCH_SIZE = 256
WINDOW_STRIDE = 128


def make_windows(frame: pd.DataFrame, window_size: int, stride: int) -> tuple[torch.Tensor, torch.Tensor]:
    windows = []
    labels = []

    for _, trial in frame.sort_values(["trial_id", "sample"]).groupby("trial_id", observed=True):
        channels = trial[IMU_CHANNELS].to_numpy(dtype=np.float32)
        if len(channels) < window_size:
            continue

        # Keep only windows that contain the trial's acceleration peak.
        acceleration_magnitude = np.linalg.norm(
            trial[ACCELERATION_CHANNELS].to_numpy(dtype=np.float32), axis=1
        )
        impact_index = int(acceleration_magnitude.argmax())
        starts = np.arange(0, len(channels) - window_size + 1, stride)
        starts = starts[(starts <= impact_index) & (impact_index < starts + window_size)]

        trial_windows = np.stack([
            channels[start:start + window_size]
            for start in starts
        ]).transpose(0, 2, 1)  # (windows, channels, time)

        windows.append(trial_windows)
        label = int(trial["label"].iloc[0])
        labels.extend([label] * len(trial_windows))

    x = torch.from_numpy(np.concatenate(windows, axis=0))
    y = torch.from_numpy(np.asarray(labels, dtype=np.int64))
    return x, y


train_x, train_y = make_windows(train_data, WINDOW_SAMPLES, WINDOW_STRIDE)
test_x, test_y = make_windows(test_data, WINDOW_SAMPLES, WINDOW_STRIDE)

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(test_x, test_y), batch_size=BATCH_SIZE, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FallConvNet(num_classes=len(fall_classes)).to(device)
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("Input shape:", tuple(train_x.shape))
print("IMU channels:", len(IMU_CHANNELS))
print("Window stride:", WINDOW_STRIDE, "samples")
print("Train windows:", len(train_y), "| Test windows:", len(test_y))
print("Device:", device)

# Training loop
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(inputs)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    correct = 0
    test_loss = 0.0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            logits = model(inputs)

            test_loss += loss_fn(logits, labels).item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()

    test_loss /= len(test_loader.dataset)
    test_accuracy = correct / len(test_loader.dataset)

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:3d}/{EPOCHS} | "
            f"train loss: {train_loss:.4f} | "
            f"test loss: {test_loss:.4f} | "
            f"test accuracy: {test_accuracy:.4f}"
        )

Input shape: (2226, 6, 200)
IMU channels: 6
Window stride: 128 samples
Train windows: 2226 | Test windows: 575
Device: cuda
Epoch   1/500 | train loss: 2.6532 | test loss: 2.6447 | test accuracy: 0.0870
Epoch  10/500 | train loss: 1.6655 | test loss: 1.6351 | test accuracy: 0.4104
Epoch  20/500 | train loss: 1.3428 | test loss: 1.3873 | test accuracy: 0.4800
Epoch  30/500 | train loss: 1.1428 | test loss: 1.2708 | test accuracy: 0.5339
Epoch  40/500 | train loss: 0.9516 | test loss: 1.2690 | test accuracy: 0.5217
Epoch  50/500 | train loss: 0.7770 | test loss: 1.2437 | test accuracy: 0.5461
Epoch  60/500 | train loss: 0.6610 | test loss: 1.2773 | test accuracy: 0.5548
Epoch  70/500 | train loss: 0.5711 | test loss: 1.3386 | test accuracy: 0.5530
Epoch  80/500 | train loss: 0.4289 | test loss: 1.3698 | test accuracy: 0.5461
Epoch  90/500 | train loss: 0.3840 | test loss: 1.6299 | test accuracy: 0.5339
Epoch 100/500 | train loss: 0.3039 | test loss: 1.5646 | test accuracy: 0.5235
Epoch 1